# Lab - Schema-Valid Output from Claude

Three tasks on making Claude answer in a fixed shape - and on what that shape does not
promise.

| Task | What you learn |
| --- | --- |
| 1 | A JSON schema describes the data you want |
| 2 | `output_config.format` makes the reply JSON that fits it, every time |
| 3 | Schema-valid is not the same as correct |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The four reasons ShopAssist can route a return on.
REASONS = ["too_small", "too_big", "damaged", "other"]

EMAILS = [
    "Hi, I want to return the blue jacket from order ORD-12345678. "
    "It arrived last week but it is too small. Alex",
    "Hello, the shoes I got are damaged. Please tell me how to send "
    "them back. Thanks, Sam",
]

---

## Task 1 - Describe the shape with a JSON schema

In [ ]:
# ============================================================
# TASK 1 - Describe the shape with a JSON schema
# ============================================================
#
# WHAT TO DO
#   Finish the schema: three properties, a reason limited to the
#   four allowed values, all three required, nothing else
#   allowed.
#
# WHY IT MATTERS
#   A support system does not want prose; it wants fields it can
#   store. Plain text answers are worded differently every time,
#   and the code reading them breaks on the first surprise. So
#   we describe the shape we want, and ask Claude to fill it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The shape is a JSON schema".
#
# HOW TO DO IT
#   Three blanks:
#
#       REASONS                         the list from the setup cell
#       ["order_id", "item", "reason"]  all three are required
#       False                           no other fields allowed
# ============================================================

# BEGIN SOLUTION
SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": "string"},
        "item": {"type": "string"},
        "reason": {"type": "string", "enum": REASONS},
    },
    "required": ["order_id", "item", "reason"],
    "additionalProperties": False,
}
# SCAFFOLD: SCHEMA = {
# SCAFFOLD:     "type": "object",
# SCAFFOLD:     "properties": {
# SCAFFOLD:         "order_id": {"type": "string"},
# SCAFFOLD:         "item": {"type": "string"},
# SCAFFOLD:         "reason": {"type": "string", "enum": ...},   # REASONS - the four allowed values
# SCAFFOLD:     },
# SCAFFOLD:     "required": ...,                 # ["order_id", "item", "reason"]
# SCAFFOLD:     "additionalProperties": ...,     # False - no other fields allowed
# SCAFFOLD: }
# END SOLUTION: replace each ... below with the value named beside it

print(json.dumps(SCHEMA, indent=2, default=str))

check("schema", schema=SCHEMA)

---

## Task 2 - Ask for output that fits the schema

In [ ]:
# ============================================================
# TASK 2 - Ask for output that fits the schema
# ============================================================
#
# WHAT TO DO
#   Send the first email, and in output_config pass the format:
#   a JSON schema, the one you just wrote. Then parse the text
#   block with json.loads.
#
# WHY IT MATTERS
#   With the format set, Claude's reply is guaranteed to be JSON
#   that matches the schema. You take the text block and parse
#   it - and it parses every time.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the request".
#
# HOW TO DO IT
#   Three blanks:
#
#       "json_schema"   the format type, in quotes
#       SCHEMA          the schema from Task 1
#       block.text      the JSON text json.loads turns into a
#                       dictionary
# ============================================================


def extract(email):
    # BEGIN SOLUTION
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        output_config={
            "effort": "low",
            "format": {"type": "json_schema", "schema": SCHEMA},
        },
        messages=[{
            "role": "user",
            "content": "Extract the return request from this customer email.\n\n" + email,
        }],
    )
    for block in response.content:
        if block.type == "text":
            return response, json.loads(block.text)
    # SCAFFOLD: response = client.messages.create(
    # SCAFFOLD:     model=MODEL,
    # SCAFFOLD:     max_tokens=300,
    # SCAFFOLD:     output_config={
    # SCAFFOLD:         "effort": "low",
    # SCAFFOLD:         "format": {"type": ..., "schema": ...},   # "json_schema", and SCHEMA
    # SCAFFOLD:     },
    # SCAFFOLD:     messages=[{
    # SCAFFOLD:         "role": "user",
    # SCAFFOLD:         "content": "Extract the return request from this customer email.\n\n" + email,
    # SCAFFOLD:     }],
    # SCAFFOLD: )
    # SCAFFOLD: for block in response.content:
    # SCAFFOLD:     if block.type == "text":
    # SCAFFOLD:         return response, json.loads(...)   # block.text - the JSON text to parse
    # END SOLUTION: replace each ... below with the value named beside it


try:
    first, data = extract(EMAILS[0])
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(data)

check("format", first=first, data=data)

---

## Task 3 - Valid is not the same as correct

In [ ]:
# ============================================================
# TASK 3 - Valid is not the same as correct
# ============================================================
#
# WHAT TO DO
#   Extract the second email - it never mentions an order
#   number - and write the check that would stop its order_id
#   from being stored.
#
# WHY IT MATTERS
#   The schema required a string, and a string came back: the
#   word "unknown". The schema was satisfied, and the value is
#   useless - stored, it would be a return request that points
#   at no order. A schema makes the answer well formed. Correct
#   values are a separate problem, and the application must
#   check them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Let's try two emails".
#
# HOW TO DO IT
#   One blank: what every real order ID starts with.
#
#       "ORD-"    in quotes
# ============================================================

try:
    _, second = extract(EMAILS[1])
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... in Task 2's extract(). Replace each one with the value "
                         "named beside it.") from None
    raise
print(second)


# BEGIN SOLUTION
def looks_like_order(order_id):
    return order_id.startswith("ORD-")
# SCAFFOLD: def looks_like_order(order_id):
# SCAFFOLD:     return order_id.startswith(...)   # "ORD-" - how every real order ID begins
# END SOLUTION: replace the ... below with the value named beside it


try:
    print("first email's order_id usable: ", looks_like_order(data["order_id"]))
    print("second email's order_id usable:", looks_like_order(second["order_id"]))
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("valid_not_correct", second=second, looks_like_order=looks_like_order)

---

## Done

Use a schema to make the output parse every time, and still validate the values before
you trust them. The next lab does exactly that: it checks the order ID and, when it is
wrong, sends the error back to Claude.

A note for the exam: the exam guide teaches forcing structured output through tool use -
a tool whose input schema is your shape, with `tool_choice` set to `any` or to that
tool. That is the answer to give on the exam. Current Opus, Sonnet and Fable models
reject those two settings; current code uses `output_config.format`, as here.